In [ ]:
import os, random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── 1. CONFIGURATION & PATHS ──────────────────────────────────
SAVE_DIR = Path('/content/drive/MyDrive/Model_results')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# TODO: Define your local or drive paths for images and metadata
IMAGE_DIR       = Path('path/to/images')
METADATA_CSV    = Path('path/to/metadata.csv')
GROUNDTRUTH_CSV = Path('path/to/groundtruth.csv')

# TODO: Experiment with these hyperparameters to optimize your multimodal model
CFG = dict(
    img_size      = 224,
    batch_size    = ...,
    num_workers   = 2,
    num_epochs    = ...,
    lr            = ...,
    weight_decay  = ...,
    dropout       = ...,
    test_split    = 0.2,
    seed          = 42,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

# ── 2. DATA PREPARATION (BINARY IMAGE + METADATA) ─────────────
meta = pd.read_csv(METADATA_CSV)
gt   = pd.read_csv(GROUNDTRUTH_CSV)

# Define target labels
MALIGNANT_CLASSES = ['AKIEC', 'BCC', 'MAL_OTH', 'MEL', 'SCCKA']
ORIG_CLASS_COLS = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']
gt['label'] = gt[ORIG_CLASS_COLS].idxmax(axis=1).apply(lambda x: 1 if x in MALIGNANT_CLASSES else 0)

df = meta.merge(gt[['lesion_id', 'label']], on='lesion_id', how='inner')

# TODO: Handle missing metadata values (Imputation)
# HINT: Check for NaNs in columns like 'age_approx' or 'site' and decide how to fill them.
df['age_approx'] = ...
df['site'] = ...

# TODO: Convert categorical metadata into numerical format using One-Hot Encoding
# HINT: Look into pd.get_dummies() for columns like 'site' and 'sex'.
df_encoded = ...
meta_cols = [c for c in df_encoded.columns if c.startswith(('site_', 'sex_', 'age_approx'))]
META_DIM = len(meta_cols)

# Split data by lesion_id
unique_lesions = df_encoded.drop_duplicates(subset='lesion_id')
train_l, temp_l = train_test_split(unique_lesions['lesion_id'], test_size=CFG['test_split'],
                                  stratify=unique_lesions['label'], random_state=CFG['seed'])
val_l, test_l = train_test_split(temp_l, test_size=0.50,
                                stratify=unique_lesions[unique_lesions['lesion_id'].isin(temp_l)]['label'],
                                random_state=CFG['seed'])

train_df = df_encoded[df_encoded['lesion_id'].isin(train_l)].reset_index(drop=True)
val_df   = df_encoded[df_encoded['lesion_id'].isin(val_l)].reset_index(drop=True)
test_df  = df_encoded[df_encoded['lesion_id'].isin(test_l)].reset_index(drop=True)

# ── 3. DATASET & TRANSFORMATIONS ─────────────────────────────
class ISICDatasetMM(Dataset):
    def __init__(self, dataframe, image_dir, meta_cols, transform=None):
        self.df, self.image_dir, self.meta_cols, self.transform = dataframe, Path(image_dir), meta_cols, transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = None
        for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
            p = self.image_dir / str(row['lesion_id']) / f"{row['isic_id']}{ext}"
            if p.exists():
                img = Image.open(p).convert('RGB')
                break
        if img is None: img = Image.new('RGB', (CFG['img_size'], CFG['img_size']))
        if self.transform: img = self.transform(img)

        meta_data = torch.tensor(row[self.meta_cols].values.astype(np.float32))
        return img, meta_data, int(row['label'])

# TODO: Define your training and evaluation transforms
train_tfm = transforms.Compose([...])
eval_tfm = transforms.Compose([...])

# ── 4. MODEL (Late Fusion Architecture) ──────────────────────
class ISICModelMM(nn.Module):
    def __init__(self, num_classes=2, meta_dim=META_DIM, dropout=0.4):
        super().__init__()
        # TODO: Select a backbone model from `timm`
        self.backbone = timm.create_model('...', pretrained=True, num_classes=0)
        img_dim = self.backbone.num_features

        # TODO: Design a simple neural network to process the metadata
        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, ...),
            nn.BatchNorm1d(...),
            nn.ReLU(),
            nn.Dropout(...)
        )

        # TODO: Design the combined classification head
        # HINT: The input size will be (Image features + Metadata features)
        self.head = nn.Sequential(
            nn.Linear(img_dim + ..., ...),
            nn.BatchNorm1d(...),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(..., num_classes)
        )

    def forward(self, x, m):
        x = self.backbone.forward_features(x)
        x = nn.AdaptiveAvgPool2d(1)(x).flatten(1)
        m = self.meta_net(m)

        # Concatenate features from both branches
        combined = torch.cat((x, m), dim=1)
        return self.head(combined)

# ── 5. TRAINING HELPER ───────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    model.train() if phase == 'train' else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    with torch.set_grad_enabled(phase == 'train'):
        for imgs, metas, labels in loader:
            imgs, metas, labels = imgs.to(DEVICE), metas.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs, metas)
            loss = criterion(logits, labels)

            if phase == 'train' and optimizer is not None:
                optimizer.zero_grad(); loss.backward(); optimizer.step()

            probs = torch.softmax(logits, dim=1)
            running_loss += loss.item() * imgs.size(0)
            correct += (probs.argmax(dim=1) == labels).sum().item()
            total += imgs.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    auc = roc_auc_score(all_labels, np.array(all_probs)[:, 1])
    return running_loss/total, correct/total, auc, all_labels, all_probs

# ── 6. EXECUTION ──────────────────────────────────────────────

# TODO: Implement DataLoaders
train_loader = DataLoader(...)
val_loader = DataLoader(...)
test_loader = DataLoader(...)

model = ISICModelMM(dropout=CFG['dropout']).to(DEVICE)

# TODO: Initialize your optimizer, scheduler, and loss function
optimizer = optim.AdamW(...)
scheduler = ...
criterion = nn.CrossEntropyLoss(...)

history = {'epoch':[], 'tr_auc':[], 'vl_auc':[], 'tr_loss':[], 'vl_loss':[], 'tr_acc':[], 'vl_acc':[]}
best_val_auc = 0.0

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_auc, _, _ = run_epoch(model, train_loader, criterion, optimizer, 'train')
    vl_loss, vl_acc, vl_auc, _, _ = run_epoch(model, val_loader, criterion, None, 'val')

    # TODO: Step your scheduler here if applicable
    # scheduler.step()

    for k, v in zip(history.keys(), [epoch, tr_auc, vl_auc, tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)

    print(f"--- Epoch {epoch:02d} Summary ---")
    print(f"TRAIN | Loss: {tr_loss:.4f} | Acc: {tr_acc:.4f} | AUC: {tr_auc:.4f}")
    print(f"VAL   | Loss: {vl_loss:.4f} | Acc: {vl_acc:.4f} | AUC: {vl_auc:.4f}")
    print(f"Time  | {time.time()-t0:.1f}s")

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        print(f"✅ Saving best model based on Val AUC: {best_val_auc:.4f}\n")
        torch.save(model.state_dict(), SAVE_DIR / 'best_binary_mm_model.pth')
    else:
        print("\n")

# ── 7. FINAL EVALUATION ──────────────────────────────────────
print('\n--- Final Evaluation (Multimodal) ---')

# TODO: Load your saved best model and run evaluation on the test_loader
model.load_state_dict(torch.load(...))
ts_loss, ts_acc, ts_auc, ts_labels, ts_probs = run_epoch(model, test_loader, criterion, phase='test')

ts_preds = np.array(ts_probs).argmax(axis=1)
print(f'Test AUC: {ts_auc:.4f} | Test Acc: {ts_acc:.4f}')
print(classification_report(ts_labels, ts_preds, target_names=['Benign', 'Malignant']))

# Boilerplate plotting code for Confusion Matrix
cm = confusion_matrix(ts_labels, ts_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title('Test Confusion Matrix (MM)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Boilerplate plotting code for Training History
epochs = history['epoch']
plt.figure(figsize=(15,5))
plt.subplot(1,3,1); plt.plot(epochs, history['tr_auc'], label='Train'); plt.plot(epochs, history['vl_auc'], label='Val'); plt.title('AUC'); plt.xlabel('Epoch'); plt.ylabel('AUC'); plt.legend()
plt.subplot(1,3,2); plt.plot(epochs, history['tr_loss'], label='Train'); plt.plot(epochs, history['vl_loss'], label='Val'); plt.title('Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.subplot(1,3,3); plt.plot(epochs, history['tr_acc'], label='Train'); plt.plot(epochs, history['vl_acc'], label='Val'); plt.title('Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / 'binary_mm_plots.png')
plt.show()